# Two-round optical-flow alignment QC validation

This notebook runs the same normalization, Farnebäck flow, warping, residual, coordinate-conversion, and cell-neighborhood aggregation functions used by `mif-pipeline alignment-qc`. It is intentionally read-only: it opens an existing canonical SpatialData store but does not write pipeline artifacts or modify the store.

Choose two exact aliases already present in `full_image`. Alias text is not interpreted, so you control whether the selected channels are imaging or AF acquisitions.

## Environment

Run this notebook in the same modern SpatialData environment used for `assemble-spatialdata` and install the optional OpenCV dependency with `pip install -e '.[alignment-qc]'`. The selected pyramid-level images and dense fields are materialized in memory.

In [ ]:
from pathlib import Path
import math

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from spatialdata import read_zarr

from mif_pipeline import load_config
from mif_pipeline.config import get_slide_config, resolve_channel_entries
from mif_pipeline.alignment_qc import (
    DEFAULT_FLOW_PARAMS,
    DENSE_METRIC_NAMES,
    _cell_observations,
    _channel_names,
    _image_levels,
    _materialize_channel,
    _round_summary,
    compute_flow_residual_maps,
    local_dapi_support,
    neighborhood_radii_pixels,
    normalize_percentile_image,
    sample_neighborhood_nanmedian,
    select_pyramid_level,
)

plt.rcParams['figure.dpi'] = 120

## Select the slide and two aliases

Set `PYRAMID_LEVEL` to an integer and `TARGET_RESOLUTION_UM` to `None` to force a specific stored level. Otherwise, the production selector chooses the stored level nearest the target resolution. A `2.6 µm` sampling radius produces a 3×3 window when the selected level is `2.6 µm/pixel`.

In [ ]:
CONFIG_PATH = Path('../example.yaml')
SLIDE_ID = 'SLIDE-0272'
REFERENCE_ALIAS = 'R1_DAPI'
MOVING_ALIAS = 'R2_DAPI'

TARGET_RESOLUTION_UM = 2.6
PYRAMID_LEVEL = None
LOWER_PERCENTILE = 1.0
UPPER_PERCENTILE = 99.9
SSIM_WINDOW_SIZE = 11
ZNCC_WINDOW_SIZE_UM = 75.0
ZNCC_MIN_VALID_FRACTION = 0.9
ZNCC_MIN_LOCAL_STD = 0.005
ZNCC_COMPUTE_CHUNKS = (1024, 1024)
CELL_SAMPLING_RADIUS_UM = 2.6
FLOW_PARAMS = dict(DEFAULT_FLOW_PARAMS)

# Plotting is decimated only for display; calculations use the complete selected level.
MAX_PLOT_DIMENSION = 2500
CELL_MARKER_SIZE = 2

# Optional full-resolution crop for the matched overlay/statistics figures below.
# Coordinates are in global microns. Use None for the complete selected level.
# Example: ZOOM_X_UM = (8_000, 10_000); ZOOM_Y_UM = (12_000, 14_000)
ZOOM_X_UM = None
ZOOM_Y_UM = None
ZOOM_FIGURE_DPI = 300
# Set to a path stem to save '<stem>_overlays.png', '<stem>_statistics.png', and '<stem>_zncc.png'.
ZOOM_OUTPUT_STEM = None

In [ ]:
config = load_config(CONFIG_PATH)
slide = get_slide_config(config, SLIDE_ID)

# This validates only exact alias resolution through the existing channel map.
resolve_channel_entries(config, SLIDE_ID, [REFERENCE_ALIAS, MOVING_ALIAS])

store_path = Path(slide['spatialdata']['store_path'])
if not store_path.exists():
    raise FileNotFoundError(f'Canonical SpatialData store not found: {store_path}')

sdata = read_zarr(store_path)
if 'full_image' not in sdata.images:
    raise KeyError("Canonical store is missing images['full_image']")
if 'agg_cell_labels' not in sdata.tables:
    raise KeyError("Canonical store is missing tables['agg_cell_labels']")

levels = _image_levels(sdata.images['full_image'])
available_aliases = _channel_names(levels[0][1])
missing = [alias for alias in (REFERENCE_ALIAS, MOVING_ALIAS) if alias not in available_aliases]
if missing:
    raise KeyError(f'Aliases absent from full_image: {missing}')

print(f'SpatialData store: {store_path}')
print(f'Available pyramid levels: {[name for name, _ in levels]}')
print(f'Available aliases ({len(available_aliases)}): {available_aliases}')

In [ ]:
selected = select_pyramid_level(
    levels,
    native_pixel_size_um=float(slide['pixel_size_um']),
    pyramid_level=PYRAMID_LEVEL,
    target_resolution_um=TARGET_RESOLUTION_UM,
)
level_array = selected['array']
pixel_size_x_um = float(selected['pixel_size_x_um'])
pixel_size_y_um = float(selected['pixel_size_y_um'])
radius_x, radius_y = neighborhood_radii_pixels(
    CELL_SAMPLING_RADIUS_UM,
    pixel_size_x_um=pixel_size_x_um,
    pixel_size_y_um=pixel_size_y_um,
)

level_details = {key: value for key, value in selected.items() if key != 'array'}
level_details['sampling_radius_pixels'] = {'x': radius_x, 'y': radius_y}
level_details['sampling_window_pixels'] = {
    'x': 2 * radius_x + 1,
    'y': 2 * radius_y + 1,
}
pd.Series(level_details)

In [ ]:
reference_raw = _materialize_channel(level_array, REFERENCE_ALIAS)
moving_raw = _materialize_channel(level_array, MOVING_ALIAS)

reference_normalized, reference_normalization = normalize_percentile_image(
    reference_raw,
    lower_percentile=LOWER_PERCENTILE,
    upper_percentile=UPPER_PERCENTILE,
)
moving_normalized, moving_normalization = normalize_percentile_image(
    moving_raw,
    lower_percentile=LOWER_PERCENTILE,
    upper_percentile=UPPER_PERCENTILE,
)

pd.DataFrame(
    [reference_normalization, moving_normalization],
    index=[REFERENCE_ALIAS, MOVING_ALIAS],
)

In [ ]:
plot_step = max(1, math.ceil(max(reference_raw.shape) / MAX_PLOT_DIMENSION))
plot_slice = np.s_[::plot_step, ::plot_step]
extent_um = [
    0,
    reference_raw.shape[1] * pixel_size_x_um,
    reference_raw.shape[0] * pixel_size_y_um,
    0,
]

fig, axes = plt.subplots(2, 2, figsize=(13, 11), constrained_layout=True)
panels = [
    (reference_raw, f'{REFERENCE_ALIAS} raw'),
    (moving_raw, f'{MOVING_ALIAS} raw'),
    (reference_normalized, f'{REFERENCE_ALIAS} normalized'),
    (moving_normalized, f'{MOVING_ALIAS} normalized'),
]
for ax, (image, title) in zip(axes.flat, panels):
    shown = ax.imshow(image[plot_slice], cmap='gray', extent=extent_um, origin='upper')
    ax.set_title(title)
    ax.set_xlabel('x (µm)')
    ax.set_ylabel('y (µm)')
    fig.colorbar(shown, ax=ax, shrink=0.75)
plt.show()

In [ ]:
flow_result = compute_flow_residual_maps(
    reference_normalized,
    moving_normalized,
    pixel_size_x_um=pixel_size_x_um,
    pixel_size_y_um=pixel_size_y_um,
    flow_params=FLOW_PARAMS,
    ssim_window_size=SSIM_WINDOW_SIZE,
)
dense_maps = {name: flow_result[name] for name in DENSE_METRIC_NAMES}

print(f"Flow direction: {flow_result['flow_direction']}")
print(f"Valid dense fraction: {np.mean(flow_result['valid_mask']):.3f}")

## Dense pre-warp and post-warp local ZNCC

This prototype computes one same-coordinate local correlation at every selected-level pixel. It uses OpenCV box filters for the required window sums and processes the image in chunks with a window-radius halo, so it never materializes an `(image height, image width, window height, window width)` array. The pre-warp map measures alignment as stored; the post-warp map measures how well the optical-flow correction explains the pair.

In [ ]:
def physical_odd_window(size_um, pixel_size_um):
    radius = max(1, int(np.ceil((float(size_um) / 2) / float(pixel_size_um))))
    return 2 * radius + 1


def dense_local_zncc_residual(
    reference,
    comparison,
    *,
    window_shape,
    minimum_valid_fraction=0.9,
    minimum_local_std=0.005,
    valid_mask=None,
    chunk_shape=(1024, 1024),
):
    reference = np.asarray(reference, dtype=np.float32)
    comparison = np.asarray(comparison, dtype=np.float32)
    if reference.shape != comparison.shape or reference.ndim != 2:
        raise ValueError('ZNCC inputs must be matching two-dimensional arrays.')

    window_y, window_x = map(int, window_shape)
    if window_y < 3 or window_x < 3 or window_y % 2 != 1 or window_x % 2 != 1:
        raise ValueError('ZNCC window dimensions must be odd integers >= 3.')
    radius_y, radius_x = window_y // 2, window_x // 2
    window_area = float(window_y * window_x)
    output = np.full(reference.shape, np.nan, dtype=np.float32)
    supplied_valid = None if valid_mask is None else np.asarray(valid_mask, dtype=bool)

    def box_sum(values):
        return cv2.boxFilter(
            values,
            ddepth=cv2.CV_64F,
            ksize=(window_x, window_y),
            normalize=False,
            borderType=cv2.BORDER_CONSTANT,
        )

    height, width = reference.shape
    chunk_y, chunk_x = map(int, chunk_shape)
    for y0 in range(0, height, chunk_y):
        y1 = min(y0 + chunk_y, height)
        ey0, ey1 = max(0, y0 - radius_y), min(height, y1 + radius_y)
        for x0 in range(0, width, chunk_x):
            x1 = min(x0 + chunk_x, width)
            ex0, ex1 = max(0, x0 - radius_x), min(width, x1 + radius_x)
            ref = reference[ey0:ey1, ex0:ex1]
            mov = comparison[ey0:ey1, ex0:ex1]
            valid = np.isfinite(ref) & np.isfinite(mov)
            if supplied_valid is not None:
                valid &= supplied_valid[ey0:ey1, ex0:ex1]
            weights = valid.astype(np.float32)
            ref = np.where(valid, ref, 0).astype(np.float32, copy=False)
            mov = np.where(valid, mov, 0).astype(np.float32, copy=False)

            count = box_sum(weights)
            safe_count = np.maximum(count, 1.0)
            mean_ref = box_sum(ref) / safe_count
            mean_mov = box_sum(mov) / safe_count
            var_ref = box_sum(ref * ref) / safe_count - mean_ref * mean_ref
            var_mov = box_sum(mov * mov) / safe_count - mean_mov * mean_mov
            covariance = box_sum(ref * mov) / safe_count - mean_ref * mean_mov
            np.maximum(var_ref, 0, out=var_ref)
            np.maximum(var_mov, 0, out=var_mov)
            std_ref = np.sqrt(var_ref)
            std_mov = np.sqrt(var_mov)
            denominator = std_ref * std_mov
            supported = (
                (count >= minimum_valid_fraction * window_area)
                & (std_ref >= minimum_local_std)
                & (std_mov >= minimum_local_std)
                & (denominator > 0)
            )
            correlation = np.zeros_like(covariance)
            np.divide(covariance, denominator, out=correlation, where=supported)
            residual = 1.0 - np.clip(correlation, 0.0, 1.0)
            residual[~supported] = np.nan

            cy0, cy1 = y0 - ey0, y1 - ey0
            cx0, cx1 = x0 - ex0, x1 - ex0
            output[y0:y1, x0:x1] = residual[cy0:cy1, cx0:cx1].astype(np.float32)
    return output


zncc_window_y = physical_odd_window(ZNCC_WINDOW_SIZE_UM, pixel_size_y_um)
zncc_window_x = physical_odd_window(ZNCC_WINDOW_SIZE_UM, pixel_size_x_um)
zncc_window_shape = (zncc_window_y, zncc_window_x)
prewarp_zncc_residual = dense_local_zncc_residual(
    reference_normalized,
    moving_normalized,
    window_shape=zncc_window_shape,
    minimum_valid_fraction=ZNCC_MIN_VALID_FRACTION,
    minimum_local_std=ZNCC_MIN_LOCAL_STD,
    chunk_shape=ZNCC_COMPUTE_CHUNKS,
)
postwarp_zncc_residual = dense_local_zncc_residual(
    reference_normalized,
    flow_result['warped_moving'],
    window_shape=zncc_window_shape,
    minimum_valid_fraction=ZNCC_MIN_VALID_FRACTION,
    minimum_local_std=ZNCC_MIN_LOCAL_STD,
    valid_mask=flow_result['valid_mask'],
    chunk_shape=ZNCC_COMPUTE_CHUNKS,
)
zncc_correction_gain = prewarp_zncc_residual - postwarp_zncc_residual
zncc_maps = {
    'prewarp_zncc_residual': prewarp_zncc_residual,
    'postwarp_zncc_residual': postwarp_zncc_residual,
    'zncc_correction_gain': zncc_correction_gain,
}
print(
    f'ZNCC window: {zncc_window_x} × {zncc_window_y} pixels '
    f'({zncc_window_x * pixel_size_x_um:.1f} × {zncc_window_y * pixel_size_y_um:.1f} µm)'
)
print(f'Pre-warp valid fraction: {np.mean(np.isfinite(prewarp_zncc_residual)):.3f}')
print(f'Post-warp valid fraction: {np.mean(np.isfinite(postwarp_zncc_residual)):.3f}')

In [ ]:
gain_values = zncc_correction_gain[np.isfinite(zncc_correction_gain)]
gain_limit = 1.0 if not len(gain_values) else max(float(np.percentile(np.abs(gain_values), 99)), 1e-6)
displacement_values = dense_maps['displacement_um'][np.isfinite(dense_maps['displacement_um'])]
displacement_limit = 1.0 if not len(displacement_values) else max(float(np.percentile(displacement_values, 99)), 1e-6)
fig, axes = plt.subplots(1, 4, figsize=(24, 6), constrained_layout=True)
zncc_panels = [
    (prewarp_zncc_residual, 'Pre-warp local ZNCC residual', 'inferno', 0, 1),
    (postwarp_zncc_residual, 'Post-warp local ZNCC residual', 'inferno', 0, 1),
    (zncc_correction_gain, 'ZNCC correction gain (pre − post)', 'coolwarm', -gain_limit, gain_limit),
    (dense_maps['displacement_um'], 'Optical-flow displacement (µm)', 'magma', 0, displacement_limit),
]
for ax, (image, title, cmap, vmin, vmax) in zip(axes, zncc_panels):
    shown = ax.imshow(
        image[plot_slice], cmap=cmap, vmin=vmin, vmax=vmax,
        extent=extent_um, origin='upper',
    )
    ax.set_title(title)
    ax.set_xlabel('x (µm)')
    ax.set_ylabel('y (µm)')
    fig.colorbar(shown, ax=ax, shrink=0.75)
plt.show()

In [ ]:
def symmetric_limit(values, percentile=99):
    finite = np.asarray(values)[np.isfinite(values)]
    return 1.0 if not len(finite) else max(float(np.percentile(np.abs(finite), percentile)), 1e-6)

def upper_limit(values, percentile=99):
    finite = np.asarray(values)[np.isfinite(values)]
    return 1.0 if not len(finite) else max(float(np.percentile(finite, percentile)), 1e-6)

fig, axes = plt.subplots(2, 3, figsize=(18, 11), constrained_layout=True)
flow_x_limit = symmetric_limit(dense_maps['flow_x_um'])
flow_y_limit = symmetric_limit(dense_maps['flow_y_um'])
dense_panels = [
    ('flow_x_um', 'Signed x displacement (µm)', 'coolwarm', -flow_x_limit, flow_x_limit),
    ('flow_y_um', 'Signed y displacement (µm)', 'coolwarm', -flow_y_limit, flow_y_limit),
    ('displacement_um', 'Displacement magnitude (µm)', 'magma', 0, upper_limit(dense_maps['displacement_um'])),
    ('warped_moving', 'Warped moving, normalized', 'gray', 0, 1),
    ('absolute_residual', 'Absolute normalized residual', 'inferno', 0, upper_limit(dense_maps['absolute_residual'])),
    ('structural_residual', 'Structural residual (1 - SSIM)', 'inferno', 0, upper_limit(dense_maps['structural_residual'])),
]
for ax, (name, title, cmap, vmin, vmax) in zip(axes.flat, dense_panels):
    image = flow_result[name]
    shown = ax.imshow(
        image[plot_slice],
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        extent=extent_um,
        origin='upper',
    )
    ax.set_title(title)
    ax.set_xlabel('x (µm)')
    ax.set_ylabel('y (µm)')
    fig.colorbar(shown, ax=ax, shrink=0.75)
plt.show()

## Matched high-resolution overlay and QC-statistics crop

Set `ZOOM_X_UM` and `ZOOM_Y_UM` in the configuration cell to inspect a rectangular region in global micron coordinates. Both figures below use the exact same pixel slice and physical extent. The overlay figure shows alignment correspondence directly; the statistics figure shows how that correspondence maps onto the six production QC panels. No display decimation is applied inside this crop.

In [ ]:
def micron_crop(image_shape, x_range_um=None, y_range_um=None):
    height, width = image_shape
    full_x = (0.0, width * pixel_size_x_um)
    full_y = (0.0, height * pixel_size_y_um)
    x_start_um, x_stop_um = full_x if x_range_um is None else map(float, x_range_um)
    y_start_um, y_stop_um = full_y if y_range_um is None else map(float, y_range_um)
    if x_stop_um <= x_start_um or y_stop_um <= y_start_um:
        raise ValueError('Zoom coordinate ranges must be increasing.')

    x0 = max(0, int(np.floor(x_start_um / pixel_size_x_um)))
    x1 = min(width, int(np.ceil(x_stop_um / pixel_size_x_um)))
    y0 = max(0, int(np.floor(y_start_um / pixel_size_y_um)))
    y1 = min(height, int(np.ceil(y_stop_um / pixel_size_y_um)))
    if x1 <= x0 or y1 <= y0:
        raise ValueError('Zoom coordinates do not overlap the selected pyramid level.')

    # imshow extent is [left, right, bottom, top]; reversing y displays image coordinates.
    extent = [
        x0 * pixel_size_x_um,
        x1 * pixel_size_x_um,
        y1 * pixel_size_y_um,
        y0 * pixel_size_y_um,
    ]
    return np.s_[y0:y1, x0:x1], extent, (x0, x1, y0, y1)


def magenta_green_overlay(reference, comparison):
    reference = np.clip(np.asarray(reference, dtype=np.float32), 0, 1)
    comparison = np.clip(np.asarray(comparison, dtype=np.float32), 0, 1)
    # Reference-only structure is magenta, comparison-only structure is green, overlap is white.
    return np.stack([reference, comparison, reference], axis=-1)


zoom_slice, zoom_extent_um, zoom_pixels = micron_crop(
    reference_normalized.shape,
    ZOOM_X_UM,
    ZOOM_Y_UM,
)
zoom_height = zoom_pixels[3] - zoom_pixels[2]
zoom_width = zoom_pixels[1] - zoom_pixels[0]
print(
    f'Zoom pixels: x={zoom_pixels[0]}:{zoom_pixels[1]}, y={zoom_pixels[2]}:{zoom_pixels[3]} '
    f'({zoom_width} × {zoom_height} pixels at {pixel_size_x_um:.4f} × {pixel_size_y_um:.4f} µm/px)'
)
print(
    f'Zoom extent: x={zoom_extent_um[0]:.1f}–{zoom_extent_um[1]:.1f} µm, '
    f'y={zoom_extent_um[3]:.1f}–{zoom_extent_um[2]:.1f} µm'
)

reference_zoom = reference_normalized[zoom_slice]
moving_zoom = moving_normalized[zoom_slice]
warped_zoom = flow_result['warped_moving'][zoom_slice]
before_overlay = magenta_green_overlay(reference_zoom, moving_zoom)
after_overlay = magenta_green_overlay(reference_zoom, warped_zoom)

overlay_fig, overlay_axes = plt.subplots(1, 4, figsize=(22, 6), constrained_layout=True)
overlay_panels = [
    (reference_zoom, f'Reference: {REFERENCE_ALIAS}', 'gray'),
    (moving_zoom, f'Moving: {MOVING_ALIAS}', 'gray'),
    (before_overlay, 'Before warp: reference magenta / moving green', None),
    (after_overlay, 'After warp: reference magenta / warped green', None),
]
for ax, (image, title, cmap) in zip(overlay_axes, overlay_panels):
    ax.imshow(image, cmap=cmap, vmin=0, vmax=1, extent=zoom_extent_um, origin='upper')
    ax.set_title(title)
    ax.set_xlabel('x (µm)')
    ax.set_ylabel('y (µm)')
    ax.set_aspect('equal')
overlay_fig.suptitle(
    f'High-resolution alignment overlay | {REFERENCE_ALIAS} → {MOVING_ALIAS}',
    fontsize=15,
)

statistics_fig, statistics_axes = plt.subplots(2, 3, figsize=(18, 12), constrained_layout=True)
zoom_flow_x_limit = symmetric_limit(dense_maps['flow_x_um'][zoom_slice])
zoom_flow_y_limit = symmetric_limit(dense_maps['flow_y_um'][zoom_slice])
zoom_panels = [
    ('flow_x_um', 'Signed x displacement (µm)', 'coolwarm', -zoom_flow_x_limit, zoom_flow_x_limit),
    ('flow_y_um', 'Signed y displacement (µm)', 'coolwarm', -zoom_flow_y_limit, zoom_flow_y_limit),
    ('displacement_um', 'Displacement magnitude (µm)', 'magma', 0, upper_limit(dense_maps['displacement_um'][zoom_slice])),
    ('warped_moving', 'Warped moving, normalized', 'gray', 0, 1),
    ('absolute_residual', 'Absolute normalized residual', 'inferno', 0, upper_limit(dense_maps['absolute_residual'][zoom_slice])),
    ('structural_residual', 'Structural residual (1 - SSIM)', 'inferno', 0, upper_limit(dense_maps['structural_residual'][zoom_slice])),
]
for ax, (name, title, cmap, vmin, vmax) in zip(statistics_axes.flat, zoom_panels):
    shown = ax.imshow(
        flow_result[name][zoom_slice],
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        extent=zoom_extent_um,
        origin='upper',
    )
    ax.set_title(title)
    ax.set_xlabel('x (µm)')
    ax.set_ylabel('y (µm)')
    ax.set_aspect('equal')
    statistics_fig.colorbar(shown, ax=ax, shrink=0.75)
statistics_fig.suptitle(
    f'Matched QC statistics crop | {REFERENCE_ALIAS} → {MOVING_ALIAS}',
    fontsize=15,
)

zoom_gain = zncc_correction_gain[zoom_slice]
zoom_gain_values = zoom_gain[np.isfinite(zoom_gain)]
zoom_gain_limit = 1.0 if not len(zoom_gain_values) else max(float(np.percentile(np.abs(zoom_gain_values), 99)), 1e-6)
zoom_displacement = dense_maps['displacement_um'][zoom_slice]
zoom_displacement_values = zoom_displacement[np.isfinite(zoom_displacement)]
zoom_displacement_limit = 1.0 if not len(zoom_displacement_values) else max(float(np.percentile(zoom_displacement_values, 99)), 1e-6)
zncc_zoom_fig, zncc_zoom_axes = plt.subplots(1, 4, figsize=(24, 6), constrained_layout=True)
zncc_zoom_panels = [
    (prewarp_zncc_residual[zoom_slice], 'Pre-warp ZNCC residual', 'inferno', 0, 1),
    (postwarp_zncc_residual[zoom_slice], 'Post-warp ZNCC residual', 'inferno', 0, 1),
    (zoom_gain, 'ZNCC correction gain', 'coolwarm', -zoom_gain_limit, zoom_gain_limit),
    (zoom_displacement, 'Optical-flow displacement (µm)', 'magma', 0, zoom_displacement_limit),
]
for ax, (image, title, cmap, vmin, vmax) in zip(zncc_zoom_axes, zncc_zoom_panels):
    shown = ax.imshow(
        image, cmap=cmap, vmin=vmin, vmax=vmax,
        extent=zoom_extent_um, origin='upper',
    )
    ax.set_title(title)
    ax.set_xlabel('x (µm)')
    ax.set_ylabel('y (µm)')
    ax.set_aspect('equal')
    zncc_zoom_fig.colorbar(shown, ax=ax, shrink=0.75)
zncc_zoom_fig.suptitle(
    f'Intensity-insensitive regional agreement | {REFERENCE_ALIAS} → {MOVING_ALIAS}',
    fontsize=15,
)

if ZOOM_OUTPUT_STEM is not None:
    output_stem = Path(ZOOM_OUTPUT_STEM).expanduser()
    output_stem.parent.mkdir(parents=True, exist_ok=True)
    overlay_path = output_stem.with_name(f'{output_stem.name}_overlays.png')
    statistics_path = output_stem.with_name(f'{output_stem.name}_statistics.png')
    zncc_path = output_stem.with_name(f'{output_stem.name}_zncc.png')
    overlay_fig.savefig(overlay_path, dpi=ZOOM_FIGURE_DPI, bbox_inches='tight')
    statistics_fig.savefig(statistics_path, dpi=ZOOM_FIGURE_DPI, bbox_inches='tight')
    zncc_zoom_fig.savefig(zncc_path, dpi=ZOOM_FIGURE_DPI, bbox_inches='tight')
    print(f'Saved: {overlay_path}')
    print(f'Saved: {statistics_path}')
    print(f'Saved: {zncc_path}')

plt.show()

In [ ]:
# A sparse vector overlay makes the signed direction easier to inspect.
quiver_step = max(1, math.ceil(max(reference_raw.shape) / 60))
yy, xx = np.mgrid[0:reference_raw.shape[0]:quiver_step, 0:reference_raw.shape[1]:quiver_step]
u = dense_maps['flow_x_um'][::quiver_step, ::quiver_step]
v = dense_maps['flow_y_um'][::quiver_step, ::quiver_step]

fig, ax = plt.subplots(figsize=(12, 10), constrained_layout=True)
ax.imshow(reference_normalized[plot_slice], cmap='gray', extent=extent_um, origin='upper')
ax.quiver(
    xx * pixel_size_x_um,
    yy * pixel_size_y_um,
    u,
    v,
    dense_maps['displacement_um'][::quiver_step, ::quiver_step],
    cmap='magma',
    angles='xy',
    scale_units='xy',
    scale=1,
    width=0.002,
)
ax.set_title(f'Reference-to-moving flow: {REFERENCE_ALIAS} → {MOVING_ALIAS}')
ax.set_xlabel('x (µm)')
ax.set_ylabel('y (µm)')
plt.show()

## Aggregate dense measurements at reference cell centers

The production stage obtains cell IDs and micron-space centers from `agg_cell_labels`, projects them onto the selected flow grid, and takes the neighborhood `nanmedian`. DAPI support is the unnormalized local moving/reference intensity ratio over the same neighborhood.

In [ ]:
source_obs, instance_ids, spatial_um = _cell_observations(sdata.tables['agg_cell_labels'])
x_level = spatial_um[:, 0] / pixel_size_x_um
y_level = spatial_um[:, 1] / pixel_size_y_um

cell_metrics = {
    name: sample_neighborhood_nanmedian(
        dense_maps[name],
        x_level,
        y_level,
        radius_x=radius_x,
        radius_y=radius_y,
    )
    for name in DENSE_METRIC_NAMES
}
cell_metrics.update(
    {
        name: sample_neighborhood_nanmedian(
            values,
            x_level,
            y_level,
            radius_x=radius_x,
            radius_y=radius_y,
        )
        for name, values in zncc_maps.items()
    }
)
cell_metrics['dapi_support'] = local_dapi_support(
    reference_raw,
    moving_raw,
    x_level,
    y_level,
    radius_x=radius_x,
    radius_y=radius_y,
    reference_dynamic_range=reference_normalization['normalization_dynamic_range'],
)

cell_qc = pd.DataFrame(
    {
        'instance_id': instance_ids,
        'x_um': spatial_um[:, 0],
        'y_um': spatial_um[:, 1],
        **cell_metrics,
    }
).set_index('instance_id')
cell_qc.head()

In [ ]:
cell_qc.describe(percentiles=[0.05, 0.5, 0.95]).T

In [ ]:
cell_panels = [
    ('flow_x_um', 'Cell flow x (µm)', 'coolwarm', True),
    ('flow_y_um', 'Cell flow y (µm)', 'coolwarm', True),
    ('displacement_um', 'Cell displacement (µm)', 'magma', False),
    ('absolute_residual', 'Cell absolute residual', 'inferno', False),
    ('structural_residual', 'Cell structural residual', 'inferno', False),
    ('dapi_support', 'Cell DAPI support ratio', 'viridis', False),
]
fig, axes = plt.subplots(2, 3, figsize=(18, 11), constrained_layout=True)
for ax, (name, title, cmap, diverging) in zip(axes.flat, cell_panels):
    values = cell_qc[name].to_numpy(dtype=float)
    finite = values[np.isfinite(values)]
    if diverging:
        limit = 1.0 if not len(finite) else max(np.percentile(np.abs(finite), 99), 1e-6)
        vmin, vmax = -limit, limit
    else:
        vmin = 0 if name != 'dapi_support' else (0 if not len(finite) else np.percentile(finite, 1))
        vmax = 1 if not len(finite) else max(np.percentile(finite, 99), vmin + 1e-6)
    ax.imshow(
        reference_normalized[plot_slice],
        cmap='gray',
        extent=extent_um,
        origin='upper',
        alpha=0.35,
    )
    shown = ax.scatter(
        cell_qc['x_um'],
        cell_qc['y_um'],
        c=values,
        s=CELL_MARKER_SIZE,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        linewidths=0,
    )
    ax.set_title(title)
    ax.set_xlabel('x (µm)')
    ax.set_ylabel('y (µm)')
    ax.set_aspect('equal')
    fig.colorbar(shown, ax=ax, shrink=0.75)
plt.show()

In [ ]:
cell_gain = cell_qc['zncc_correction_gain'].to_numpy(dtype=float)
cell_gain_values = cell_gain[np.isfinite(cell_gain)]
cell_gain_limit = 1.0 if not len(cell_gain_values) else max(float(np.percentile(np.abs(cell_gain_values), 99)), 1e-6)
cell_zncc_panels = [
    ('prewarp_zncc_residual', 'Cell pre-warp ZNCC residual', 'inferno', 0, 1),
    ('postwarp_zncc_residual', 'Cell post-warp ZNCC residual', 'inferno', 0, 1),
    ('zncc_correction_gain', 'Cell ZNCC correction gain', 'coolwarm', -cell_gain_limit, cell_gain_limit),
]
fig, axes = plt.subplots(1, 3, figsize=(20, 7), constrained_layout=True)
for ax, (name, title, cmap, vmin, vmax) in zip(axes, cell_zncc_panels):
    ax.imshow(
        reference_normalized[plot_slice], cmap='gray', extent=extent_um,
        origin='upper', alpha=0.35,
    )
    shown = ax.scatter(
        cell_qc['x_um'], cell_qc['y_um'], c=cell_qc[name],
        s=CELL_MARKER_SIZE, cmap=cmap, vmin=vmin, vmax=vmax, linewidths=0,
    )
    ax.set_title(title)
    ax.set_xlabel('x (µm)')
    ax.set_ylabel('y (µm)')
    ax.set_aspect('equal')
    fig.colorbar(shown, ax=ax, shrink=0.75)
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 13), constrained_layout=True)
for ax, name in zip(axes.flat, cell_metrics):
    values = cell_qc[name].to_numpy(dtype=float)
    values = values[np.isfinite(values)]
    if len(values):
        lower, upper = np.percentile(values, [0.5, 99.5])
        shown = values[(values >= lower) & (values <= upper)]
        ax.hist(shown, bins=80, color='steelblue', alpha=0.85)
    ax.set_title(name)
    ax.set_ylabel('cells')
plt.show()

In [ ]:
# This is the same compact summary constructor used by the production stage.
summary = _round_summary(
    alias=MOVING_ALIAS,
    index=1,
    is_reference=False,
    normalization=moving_normalization,
    maps=dense_maps,
    support=cell_metrics['dapi_support'],
)
pd.Series(summary, name=MOVING_ALIAS).to_frame()

## Interpretation notes

- Flow is stored in the reference-to-moving direction. The moving image is sampled at `reference coordinate + flow` to warp it into reference coordinates.
- Pre-warp ZNCC residual measures regional agreement in the stored images while ignoring local positive gain and offset. Post-warp ZNCC residual measures how well the optical-flow correction explains the pair.
- A large positive ZNCC correction gain together with coherent displacement is the expected signature of a correctable shifted region.
- High displacement alone does not prove valid correspondence; inspect residual and DAPI-support maps together.
- Structural residual is `1 - local SSIM`, a general unexplained-disagreement measure rather than a tissue-loss probability.
- DAPI support uses unnormalized intensities so a globally weak acquisition cannot be rescued solely by independent percentile normalization.
- This notebook makes no filtering decisions and writes nothing. After choosing production settings, run the explicit `alignment-qc` stage separately.